# Welcome to the QAML Challenge

**CSAW 2026 Quantum Security Challenge · materials and submissions open 11 November 2026**

Learn how bounded changes to images affect a quantum classifier, then explore
defenses that make it more robust. All examples use the supplied offline Road
Signs benchmark. You need Python 3.11 and a CPU; no quantum hardware is required.

**Before running:** follow [setup](docs/setup.md). Run cells from top to bottom.
The local examples take small batches so you can understand each step. The
black-box request is opt-in and uses your group's metered allowance.

| Your choice | Model access | What you submit |
|---|---|---|
| Attack a hardened QNN or QCNN, white-box | Architecture, encoding, measurements, head, weights and gradients | Five NPZ files, source and write-up |
| Attack a hardened QNN or QCNN, black-box | Private group's prediction token; four probabilities | Five NPZ files, source, query accounting and write-up |
| Defend an original QNN or QCNN | Original model, approved labeled training data | Reproducible defense protocol or training recipe, source and write-up |

For attacks, choose **targeted** (the supplied incorrect target class) or
**untargeted** (any incorrect class). Submit each family/access/objective
separately. The local examples are starting points, not complete solutions.

Challenge day is **13 November**, GST for MENA and ET for US–Canada. Check the
[schedule](https://github.com/walid-mk/csaw-quantum-challenges-2026/blob/main/docs/schedule.md)
for regional times. **Submit by Friday, 13 November, 6:00 PM GST (MENA) or 6:00 PM ET
(US–Canada, online). Start solving from the 11 November release.** Keep your solution private until both editions finish.

## Background: what are you exploring?

An **adversarial example** is an input deliberately changed to cause an incorrect
prediction. Here, the input is a road-sign image. If `x` is the original and
`delta` is the change, the model sees `x + delta`. The default **L-infinity**
bound requires `max(abs(delta)) <= 8/255`: each red, green and blue channel at
each pixel may move by at most eight intensity levels on a 0–255 scale. This
measures channel changes, not the number of changed pixels.

A **quantum adversarial attack** targets a classifier with a quantum component.
In this challenge, classical preprocessing converts the image into eight angles;
rotation gates encode those angles, a parameterized circuit processes them,
measurements become features, and a classical head outputs four scores. Changing
the image changes those angles and can change the prediction. No attack on a
physical quantum device is involved.

- **Untargeted:** make the prediction incorrect, regardless of which wrong class
  is returned. A stop sign predicted as either crosswalk or traffic light qualifies
  if the reference label is stop.
- **Targeted:** make the prediction equal the specific supplied incorrect target.
  If that target is crosswalk, a traffic-light prediction does not qualify.
- **White-box:** you can inspect architecture, weights and input gradients.
- **Black-box:** you query the assigned endpoint and receive probabilities,
  subject to your group's allowance.
- **Defense:** improve the original model's reliability, through training or an
  added input transformation. Measure performance on both clean and attacked inputs.
- **Adaptive evaluation:** attacks are constructed against the defended pipeline,
  including its preprocessing, rather than only against the original model.
- **Logits vs probabilities:** logits are unnormalized class scores; softmax
  converts them to nonnegative probabilities summing to one. The largest output
  selects the predicted class. High confidence alone does not establish correctness.

These exercises use approved offline images and model simulations. You need only
basic Python, tensors, loss functions and gradients; the following cells provide
small running examples.


## 1. Load and verify your kit

Pixels are RGB, channels-first, shape `(N, 3, 32, 32)`, `float64`, in `[0,1]`.
Do not resize or refit preprocessing. The model performs the supplied
preprocessing → eight RY angles → quantum circuit → measurements → linear head.

The class order is **speedlimit, crosswalk, trafficlight, stop**. Trafficlight
means a physical traffic light. Training labels are available; evaluation labels
are not. The benchmark uses public/exposed Road Signs images.

In [1]:
# Output: kit status and verified file count; an assigned team ID appears only if TEAM.json exists.
from pathlib import Path
import os
import json
import sys
import numpy as np
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'kit/sdk.py').is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from kit.sdk import verify_assets, load_model, load_inputs, write_submission
from client.scoring import qnn_defense_points, qcnn_defense_points, attack_points, feedback_summary

torch.set_num_threads(1)
torch.manual_seed(2026)  # Reproduce this local demonstration.
manifest = verify_assets()
CLASSES = ['speedlimit', 'crosswalk', 'trafficlight', 'stop']
POLICY = json.loads((ROOT / 'config/policy.json').read_text())
cap = POLICY['pixel_cap']
EPSILON = cap['numerator'] / cap['denominator'] if cap['enabled'] else 1.0
print('Kit:', manifest['status'], '| verified files:', len(manifest['files']))
if (ROOT / "TEAM.json").exists():
    team = json.loads((ROOT / "TEAM.json").read_text())
    print("Your team ID:", team["team_id"], "| region:", team["region"])

Kit: released | verified files: 45


## 2. Choose a model and inspect its outputs

Set `FAMILY` to `qnn` or `qcnn`; use runs 0–4. Use **hardened** for attack work
and **original** for defense development. The QNN uses three native circuit
layers and 16 measurements; the QCNN uses SU4 convolution/pooling with
8→4→2→1 active wires and X/Y/Z readout. Both have a classical four-class head.

`model(images)` returns logits. Apply `softmax` for class probabilities.

In [2]:
# Output: one RGB image shape, four probabilities, assigned target, and classical head dimensions.
FAMILY = 'qnn'             # Change to 'qcnn' to explore the second family.
OBJECTIVE = 'targeted'     # Or 'untargeted'.
RUN = 0                   # 0, 1, 2, 3, 4
model = load_model(FAMILY, run=RUN, kind='hardened')
batch = load_inputs(FAMILY, 'whitebox')
x0 = torch.from_numpy(batch['images'][:1].copy())
with torch.no_grad():
    p0 = model(x0).softmax(-1)
print('Input:', tuple(x0.shape), x0.dtype)
print('Probabilities:', dict(zip(CLASSES, p0[0].tolist())))
print('Assigned target:', CLASSES[int(batch['targets'][0])])
print('Quantum features:', model.head.in_features, '| class outputs:', model.head.out_features)

Input: (1, 3, 32, 32) torch.float64
Probabilities: {'speedlimit': 0.9911285686307738, 'crosswalk': 0.0032068218388055193, 'trafficlight': 0.003518773126038741, 'stop': 0.00214583640438195}
Assigned target: crosswalk
Quantum features: 16 | class outputs: 4


You can inspect the full white-box circuit and head. The following example draws
the actual circuit used for this input. See `kit/src/qaml_ctf/` for model source.

In [3]:
# Output: the actual quantum gates/readout and the linear classifier; this cell does not train anything.
import pennylane as qml
angles = model.preprocess(x0)[0]
circuit_args = ((angles, model.quantum_weights) if FAMILY == 'qnn'
                else (angles, model.conv_weights, model.pool_weights))
print(qml.draw(model.qnode)(*circuit_args))
print(model.head)

0: ──RY(0.72)───RY(0.66)───RZ(0.01)───RY(0.80)───RZ(0.01)───RY(0.88)───RZ(-0.18)─────────── ···
1: ──RY(-0.93)──RY(0.35)───RZ(0.70)──╭●──────────RY(0.27)───RZ(1.10)──╭●──────────RY(0.07)─ ···
2: ──RY(-0.31)──RY(0.49)───RZ(0.15)──╰Z──────────RY(0.29)───RZ(0.15)──╰Z──────────RY(0.75)─ ···
3: ──RY(-0.34)──RY(-1.14)──RZ(0.38)──╭●──────────RY(-0.54)──RZ(-0.37)─╭●──────────RY(-0.10) ···
4: ──RY(-0.31)──RY(-0.32)──RZ(-0.28)─╰Z──────────RY(-1.29)──RZ(-0.45)─╰Z──────────RY(-0.40) ···
5: ──RY(-0.62)──RY(0.18)───RZ(0.03)──╭●──────────RY(-0.50)──RZ(0.94)──╭●──────────RY(-0.45) ···
6: ──RY(-0.54)──RY(-0.49)──RZ(0.52)──╰Z──────────RY(-0.76)──RZ(0.08)──╰Z──────────RY(-0.47) ···
7: ──RY(-0.84)──RY(-0.09)──RZ(0.20)───RY(-0.21)──RZ(-0.34)──RY(-0.18)──RZ(0.01)──────────── ···

0: ··· ───────────────┤  <Z> ╭<Z@Z>                                           ╭<Z@Z>
1: ··· ──RZ(-0.10)─╭●─┤  <Z> ╰<Z@Z> ╭<Z@Z>                                    │     
2: ··· ──RZ(-0.14)─╰Z─┤  <Z>        ╰<Z@Z> ╭<Z@Z>            

## 3. White-box attack: one bounded update

The default policy allows a change of at most **8/255 in any pixel channel**, measured against
the supplied original. An image outside this bound is invalid even if it fools
the model. Always project candidates back to that original-relative range. Check
[the published policy](config/policy.json) and announcements before submission;
only organizers can change limits.

This single-step example follows the input gradient. For a targeted attack it
reduces loss for the assigned target. For untargeted exploration it increases
loss for the model's current prediction, used here as a **pseudo-label** because
evaluation labels are withheld. A changed prediction alone is not an official
success measurement. Try your own permitted method and compare all five runs.

In [4]:
# Output: largest original-relative change and predictions before/after; this is a local demonstration, not a score.
# YOUR ATTACK: replace the update/search logic below, retaining original-relative projection.
x = x0.clone().requires_grad_(True)
chosen_label = (int(batch['targets'][0]) if OBJECTIVE == 'targeted' else int(p0.argmax(-1)[0]))
loss = torch.nn.functional.cross_entropy(model(x), torch.tensor([chosen_label]))
gradient = torch.autograd.grad(loss, x)[0]
direction = -1 if OBJECTIVE == 'targeted' else 1
candidate = x.detach() + direction * EPSILON * gradient.sign()
candidate = torch.maximum(torch.minimum(candidate, x0 + EPSILON), x0 - EPSILON).clamp(0, 1)
delta = float((candidate - x0).abs().max())
assert torch.isfinite(candidate).all() and delta <= EPSILON + 1e-12
with torch.no_grad():
    p1 = model(candidate).softmax(-1)
print('Maximum change:', delta, '| allowed:', EPSILON)
print('Before:', CLASSES[int(p0.argmax(-1)[0])], '| after:', CLASSES[int(p1.argmax(-1)[0])])
print('Chosen-class probability:', float(p0[0, chosen_label]), '→', float(p1[0, chosen_label]))

Maximum change: 0.03137254901960787 | allowed: 0.03137254901960784
Before: speedlimit | after: speedlimit
Chosen-class probability: 0.0032068218388055193 → 0.07603927914663257


## 4. Black-box access

Your group receives **one token** for all black-box variants. Use `prediction_url`
to choose the family, objective and run. Teammates share the default allowance of **273
queries per image per variant/run**. The service response identifies the active
policy; `budget: null` means an organizer has explicitly disabled the quota. Responses contain probabilities and usage;
there are no gradients or weights. A batch costs one query per image. A timed-out
request may already be charged, so the client never retries automatically.

Set the switch below to `True` only when you want to spend a query and have the
issued URL and token. Paste the token only into the hidden prompt, never into a
code cell. If it is already configured, the cell reads `QAML_PREDICT_URL` and
`QAML_QUERY_TOKEN` from your environment. See [access details](docs/black-box-access.md).

In [5]:
# Output: probabilities and cumulative grant usage, or the opt-in message. Never print group_token.
RUN_BLACKBOX_REQUEST = False
if RUN_BLACKBOX_REQUEST:
    import getpass
    from client.access import prediction_url
    from client.prediction import predict, PredictionError
    base_url = os.environ.get('QAML_PREDICT_URL') or input('Issued service URL: ').strip()
    group_token = os.environ.get('QAML_QUERY_TOKEN') or getpass.getpass('Group token: ')
    try:
        # YOUR BLACK-BOX ATTACK: propose candidates and track each image\'s calls here.
        bb = load_inputs(FAMILY, 'blackbox')
        reply = predict(prediction_url(base_url, FAMILY, OBJECTIVE, RUN), group_token,
                        bb['ids'][:1].tolist(), bb['images'][:1].tolist())
        print('Probabilities:', dict(zip(CLASSES, reply['probabilities'][0])))
        print('Grant usage:', reply['queries'], '/', reply['budget'])
    except PredictionError as error:
        print(str(error))  # The client suppresses credentials and response bodies.
    finally:
        del group_token
else:
    print('No black-box query requested. Enable the switch when you have group access.')

No black-box query requested. Enable the switch when you have group access.


## 5. Defense: implement your protocol and declare a recipe

**Adversarial training is an example, not a requirement.** You may explore
preprocessing, alternative training objectives, regularization, or your own
methods. Retain the assigned quantum architecture, encoding, measurements,
fixed model preprocessing and classical head. An added image transform runs
*before* that fixed preprocessing. Use only the supplied training labels.

There are two submission interfaces:

- The compact six-field adversarial-training recipe, demonstrated below.
- A `custom_protocol` recipe with `protocol.py`: implement `fit` for optional
  training and `preprocess` for an optional image transform. Identity transforms
  and no-training defenses are valid. The organizer reproduces all five runs.

Read [custom defense protocols](docs/defense-protocols.md) for the callable
interface, reproducibility requirements and adaptive evaluation. A method that
hides gradients still faces adaptive attacks. Disclose any randomness and
non-differentiable operations so they can be evaluated appropriately.

The next cell illustrates **one update on four training images** using the
compact recipe. It changes only the in-memory model. Its training accuracy is
not validation accuracy or a competition score. You may replace this training
logic in your own custom protocol; the original checkpoint files stay unchanged.


In [6]:
# Output: one minibatch loss and tiny training BA; neither estimates held-out robustness.
# YOUR DEFENSE: this is one AT example; custom protocols can use a different training method.
recipe = {'family': FAMILY, 'method': 'adversarial_training',
          'radius_numerator': 4, 'adversarial_fraction': 0.25,
          'attack_steps': 1, 'learning_rate': 0.005}
with np.load(ROOT / 'kit/training/road-signs-training.npz', allow_pickle=False) as training:
    # One training example per class, for this small mechanics demonstration.
    indices = [int(np.flatnonzero(training['labels'] == c)[0]) for c in range(4)]
    train_x = torch.from_numpy(training['images'][indices].copy())
    train_y = torch.from_numpy(training['labels'][indices].copy()).long()
defender = load_model(FAMILY, run=RUN, kind='original')
optimizer = torch.optim.Adam(defender.parameters(), lr=recipe['learning_rate'])
n_adv = int(len(train_x) * recipe['adversarial_fraction'])
origin = train_x[:n_adv]
adv = origin.clone()
train_eps = recipe['radius_numerator'] / 255
for _ in range(recipe['attack_steps']):
    adv = adv.detach().requires_grad_(True)
    adv_loss = torch.nn.functional.cross_entropy(defender(adv), train_y[:n_adv])
    grad = torch.autograd.grad(adv_loss, adv)[0]
    adv = adv.detach() + (train_eps / recipe['attack_steps']) * grad.sign()
    adv = torch.maximum(torch.minimum(adv, origin + train_eps), origin - train_eps).clamp(0, 1)
mixed = train_x.clone()
mixed[:n_adv] = adv.detach()
optimizer.zero_grad()
training_loss = torch.nn.functional.cross_entropy(defender(mixed), train_y)
training_loss.backward()
optimizer.step()
with torch.no_grad():
    training_prediction = defender(train_x).argmax(-1)
    recalls = [(training_prediction[train_y == c] == c).double().mean().item() for c in range(4)]
print('One-step training loss:', float(training_loss.detach()))
print('Tiny training balanced accuracy (not validation):', np.mean(recalls))
print('Recipe:', recipe)

One-step training loss: 0.19299197719118277
Tiny training balanced accuracy (not validation): 1.0
Recipe: {'family': 'qnn', 'method': 'adversarial_training', 'radius_numerator': 4, 'adversarial_fraction': 0.25, 'attack_steps': 1, 'learning_rate': 0.005}


### Alternative example: an input preprocessing defense

This example blends an image with a local average before calling the unchanged
original quantum model. It requires no training. It illustrates the custom
interface; it is not a claim that smoothing is effective. Replace `fit` and/or
`preprocess` in your own `protocol.py`, and explain the method in your write-up.
The output shows four probabilities and confirms that the input gradient can
pass through the added transform for adaptive evaluation.


In [7]:
# Output: defended probabilities and an input-gradient norm, not a robustness score.
import importlib.util
from client.defense import DefendedModel
example_path = ROOT / 'examples/custom-defense/protocol.py'
spec = importlib.util.spec_from_file_location('defense_example', example_path)
protocol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(protocol)
recipe = json.loads((example_path.parent / 'recipe.json').read_text())
recipe['family'] = FAMILY
# YOUR DEFENSE: implement fit/preprocess in your own protocol.py and declare its config.
base = load_model(FAMILY, run=RUN, kind='original')
state = protocol.fit(base, train_x, train_y, seed=2026, config=recipe['config'])
defended = DefendedModel(base, protocol.preprocess, state=state,
                         config=recipe['config'], backward=recipe['backward'])
probe = train_x[:1].clone().requires_grad_(True)
logits = defended(probe)
dx = torch.autograd.grad(logits.sum(), probe)[0]
print('Preprocessed-defense probabilities:', logits.softmax(-1).detach().tolist())
print('Input-gradient norm:', float(dx.norm()))


Preprocessed-defense probabilities: [[0.9751215496894384, 0.0031544588848978327, 0.02168541783145341, 3.8573594210266974e-05]]
Input-gradient norm: 0.36949938215348455


## 6. Understand scores and read feedback

**Balanced accuracy** is mean per-class recall, so each class counts equally.
For defense, `C` is clean balanced accuracy and `R` is robust balanced accuracy
under the full attack suite. They are fractions in `[0,1]`. A tiny training batch
or a single attack does not measure the full evaluation.

QNN defense: `100 × clip((R − R0)/(1 − R0)) × clip(C/C0)`, with
`R0 = 0.5958333333333333`, `C0 = 0.9003793261336365`, and `clip` to `[0,1]`.
There are no clean eligibility cutoffs for QNN.

QCNN defense: clean balanced accuracy must be at least `0.7915030824420426`,
and the upper 95% confidence bound on paired clean loss at most `0.06671775562793336`.
Points are `100 × clip(L / 0.12332663690476187)`, where `L` is the lower 95%
confidence bound on paired robust gain; otherwise points are zero. Positive
robust gain requires `L > 0`.

Attack: `100 × weight × clip(L / anchor)`, using the lower 95% confidence bound
on success. Targeted weight is 1.25; untargeted weight is 1. Anchors are in
[scoring settings](config/scoring.json). Missing/invalid records remain failures
in the original denominator. You need the grader's metric to reproduce points;
probabilities alone do not determine your score.

The following are **illustrative calculations**, not submitted or awarded scores.

In [8]:
# Output: illustrative points calculated from the stated metrics, not awarded competition scores.
from client.scoring import SETTINGS
baseline = SETTINGS['qnn_defense']['baseline']
examples = {
    'QNN unchanged baseline': qnn_defense_points(float(baseline['R0']), float(baseline['C0'])),
    'QNN hypothetical R=.65, C=.90': qnn_defense_points(.65, .90),
    'QCNN hypothetical C=.85, clean loss upper=.02, gain lower=.05': qcnn_defense_points(.85, .02, .05),
    'QNN white-box targeted hypothetical success lower=.02': attack_points('qnn', 'whitebox', 'targeted', .02),
}
for label, points in examples.items():
    print(f'{label}: {points:.4f} points')

QNN unchanged baseline: 0.0000 points
QNN hypothetical R=.65, C=.90: 13.3964 points
QCNN hypothetical C=.85, clean loss upper=.02, gain lower=.05: 40.5427 points
QNN white-box targeted hypothetical success lower=.02: 42.8731 points


After grading, save your returned feedback as JSON and set its path below. A
`not_scored` result has null points, not zero. QNN invalid/missing submissions
score zero with causes; evaluator failures are retried at most three times with
the same inputs, then reported as not scored. See [result causes](docs/scoring-causes.md).
QNN and QCNN defense standings are separate; attack variants also stay identified.

In [9]:
# Output: your saved grading result, or instructions to set a feedback path.
FEEDBACK_PATH = None  # Example: Path('submissions/defense-001/feedback.json')
if FEEDBACK_PATH is not None:
    result = json.loads(Path(FEEDBACK_PATH).read_text())
    print(json.dumps(feedback_summary(result), indent=2))
else:
    print('Set FEEDBACK_PATH after receiving your graded result.')

Set FEEDBACK_PATH after receiving your graded result.


## 7. Package an attack or defense

The next cell writes **practice files** under ignored `practice-output/`. It
demonstrates the format; unchanged images are not an attack solution. For a real
attack, compute your own candidates for **every image and all five runs** of one
variant. Preserve the exact ID order, and save only `ids` and `images` in each NPZ.
For a defense, submit your recipe and, for a custom protocol, `protocol.py` and
reproduction instructions. Organizers reproduce model weights from your code;
there is no opaque checkpoint-only submission.

In [10]:
# Output: a new practice directory containing five format examples and a defense recipe.
import tempfile
(ROOT / 'practice-output').mkdir(exist_ok=True)
practice = Path(tempfile.mkdtemp(prefix='walkthrough-', dir=ROOT / 'practice-output'))
PACKAGE_ACCESS = 'whitebox'  # Change to 'blackbox' for those supplied input IDs.
format_inputs = load_inputs(FAMILY, PACKAGE_ACCESS)
for run in range(5):
    # Replace these unchanged copies with candidates computed for each target run.
    output = practice / 'attack' / f'run-{run}.npz'
    write_submission(output, format_inputs['ids'], format_inputs['images'])
    with np.load(output, allow_pickle=False) as saved:
        assert set(saved.files) == {'ids', 'images'}
        assert np.array_equal(saved['ids'], format_inputs['ids'])
        assert saved['images'].dtype == np.float64
        assert np.max(np.abs(saved['images'] - format_inputs['images'])) <= EPSILON + 1e-12
(practice / 'defense').mkdir()
(practice / 'defense/recipe.json').write_text(json.dumps(recipe, indent=2) + '\n')
if recipe.get('method') == 'custom_protocol':
    import shutil
    shutil.copyfile(ROOT / 'examples/custom-defense/protocol.py', practice / 'defense/protocol.py')
    # Replace this example with YOUR matching protocol.py in a real submission.
print('Practice files:', practice.relative_to(ROOT))

Practice files: practice-output/walkthrough-zw3hnlla


## 8. Submit through your group's private repository

You do **not** need a pull request to the shared challenge repository. Each group
uses its own independent private repository so other groups cannot read its
code, issues or feedback. An issue is the submission notification; the committed
files are the solution.

1. Use **Use this template → Create a new repository → Private** on the public
   challenge repository. Do not fork it. Choose a name such as
   `csaw-qaml-2026-group-01`. Invite your teammates and **`walid-mk`** through
   **Settings → Collaborators → Add people**. Privately send organizers your
   group number and repository URL; wait until organizer access is accepted.
   Create `submissions/defense-001/` or a distinct attack submission directory.
2. Copy and complete [submission.json](templates/submission.json) and
   [writeup.md](templates/writeup.md). Copy `templates/TEAM.json` to `TEAM.json` and enter your organizer-assigned team ID and region.
3. For defense, use `scenario: defense`, `access: whitebox`, `objective: null`,
   a recipe path, and `attack_files: []`. For attack, select the actual access
   and objective, use `recipe: null`, and list five run paths in run order.
   All paths are relative to the repository root.
4. Push your complete source, artifacts and explanation, then obtain the SHA:

```bash
git add submissions/defense-001
git commit -m "Submit QNN defense recipe and analysis"
git push
git rev-parse HEAD
```

5. Before the regional submission cutoff, open **Issues → New issue → Submit
   QAML solution** in **your group's repository**. Include the full 40-character
   SHA and `submissions/defense-001/submission.json` path.
6. For a revision, submit a new issue with the new SHA. The grader captures the
   exact submitted commit; later edits do not replace it. Feedback arrives privately.

Keep tokens out of code, files, notebook outputs and issues. Explain your method,
results, limitations, query use, dependencies and any material AI assistance.
See [submission details](docs/submissions.md), [rules](docs/rules.md),
[scoring](docs/scoring.md). Optional hints will be provided on 13 November.